In [ ]:
import streamlit as st
from streamlit_jupyter import StreamlitPatcher, tqdm
StreamlitPatcher().jupyter()  # register streamlit with jupyter-compatible wrappers
import sys; sys.path.append('..')
from osp import *
pd.options.display.max_colwidth = 200
pd.options.display.max_rows = 20

In [ ]:
# pprint(get_sent_html(get_sent_obj("The old world is dying, and the new world struggles to be born."),show_labels=True))

In [ ]:
doc = get_nlp_doc(newtxt)
sent = doc.sentences[0]
word = sent.words[52]
word.id

In [ ]:
word_i = word.id - 1
window = 10
window_rad = window//2
context = word.sent.words[word_i-window_rad:word_i+window_rad+1]

In [ ]:
HTML(get_sent_html(context,show_labels=True,highlight_word_id=word.id))

In [ ]:
def detokenize_stanza(tokens):
    l = []
    for tok in tokens:
        l.append(tok.text)
        l.append(tok.spaces_after)
    return ''.join(l)

def get_eg_from_word(word, window=10, window_chars=100, html=True):
    word_i = word.id - 1
    sent = word.sent
    words = sent.words
    tokens = sent.tokens
    try:
        prev_tokens = tokens[:word_i]
        this_token = tokens[word_i]
        next_tokens = tokens[word_i+1:]
    except:
        return ""
    
    if window:
        window_rad = window//2
        words = words[word_i-window_rad:word_i+window_rad+1]
        tokens = tokens[word_i-window_rad:word_i+window_rad+1]
    if html:
        return get_sent_html(words,show_labels=True,highlight_word_id=word.id)
    else:
        
        midstr = f'*{word.text.upper()}*{this_token.spaces_after}'

        prev_text = detokenize_stanza(prev_tokens)[-40:]
        next_text = detokenize_stanza(next_tokens)[:60 - len(midstr)]
        return f'{prev_text}{midstr}{next_text}'


def extract_feat_examples(doc, max_per_feat=1, window=10, window_chars=100):
    sents = doc.sentences
    random.shuffle(sents)
    egs = defaultdict(list)
    feat2word2count = defaultdict(Counter)
    for sent in sents:
        for word in sent.words:
            for feat_type in ['deprel','pos']:
                feat_val = word.deprel if feat_type == 'deprel' else word.xpos
                if feat_val:
                    feat = f'{feat_type}_{feat_val}'
                    egs[feat].append(word)
                    feat2word2count[feat][word.text.lower()] += 1
    
    o = []
    for feat,words in egs.items():
        if len(words) > max_per_feat:
            words = random.sample(words, max_per_feat)
        else:
            random.shuffle(words)
        
        for word in words:
            word_str = word.text.lower()
            count = feat2word2count[feat][word_str]
            total = feat2word2count[feat].total()
            perc = round(count/total*100, 1) if total else 0
            odx = {
                'feature': feat,
                'word': word_str,
                'count': count,
                'perc': perc,
                'eg_text': get_eg_from_word(word, html=False, window=window, window_chars=window_chars),
                'eg_html': get_eg_from_word(word, html=True, window=window, window_chars=window_chars),
                'sent_id':word.sent.id,
                'word_id':word.id,
            }
            o.append(odx)
    return o
    

In [ ]:
# extract_feat_examples(doc)

In [ ]:
STASH_FEAT_EXAMPLES3 = get_stash('osp_feat_examples3',append_mode=True)
# STASH_FEAT_EXAMPLES2.clear()

In [ ]:
import multiprocessing as mp

def _do_gen_feat_examples(args):
    docstr,max_per_feat,window = args
    doc = stanza.Document.from_serialized(docstr)
    return extract_feat_examples(doc, max_per_feat=max_per_feat, window=window)

def gen_feat_examples(max_per_feat=3, window=10, force=False, num_proc=1, lim=None, batch_size=100):
    ids_done = set(STASH_FEAT_EXAMPLES2.keys())
    ids_todo = set(get_parsed_slice_ids()) - ids_done if not force else get_parsed_slice_ids()
    ids_todo = list(ids_todo)
    random.shuffle(ids_todo)
    ids_todo = ids_todo[:lim]

    if num_proc < 2:
        for id in tqdm(ids_todo):
            _do_gen_feat_examples((STASH_SLICES_NLP[id], max_per_feat, window))
        return
    
    def iter_objs():
        for id in ids_todo:
            docstr = STASH_SLICES_NLP.get(id,None)
            yield (docstr, max_per_feat, window)
    
    with mp.Pool(num_proc) as p:
        iterr = p.imap(_do_gen_feat_examples, iter_objs(), chunksize=1)
        iterr = zip(ids_todo, iterr)
        iterr = tqdm(iterr, total=len(ids_todo))
        for id,data in iterr:
            STASH_FEAT_EXAMPLES2[id] = data


In [2]:
# gen_feat_examples3(lim=100,num_proc=10)

In [3]:
# STASH_FEAT_EXAMPLES3.get_all('deprel_nsubj')

0it [00:00, ?it/s]


In [ ]:
for id,dfx in STASH_FEAT_EXAMPLES3.items():
    pass
dfx

In [ ]:
for id,docstr in STASH_SLICES_NLP.items():
    doc = stanza.Document.from_serialized(docstr)
    break

In [ ]:
sents = doc.sentences
random.shuffle(sents)
sents[0].to_dict()

In [ ]:
gen_feat_examples()

In [ ]:
extracted_examples = extract_examples()
extracted_examples


In [ ]:
slice_ids = get_slice_ids(['phil/10.2307/40231690'])

In [ ]:
get_slices_feats(['phil/10.2307/40231690'])